In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Torch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

Torch version: 2.12.1
MPS available: True


In [4]:
dataset = load_dataset("reshabhs/SPML_Chatbot_Prompt_Injection")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['System Prompt', 'User Prompt', 'Prompt injection', 'Degree', 'Source'],
        num_rows: 16012
    })
})


In [5]:
df = dataset["train"].to_pandas()

df["text"] = df["System Prompt"].fillna("") + " " + df["User Prompt"].fillna("")
df = df[["text", "Prompt injection", "Degree"]].copy()
df = df.rename(columns={"Prompt injection": "label"})

df.head()

,text,label,Degree
0,You are taking on the role of Healthcare Advis...,1,1
1,"You are a chatbot named Financial Advisor Bot,...",1,1
2,"You are the Online Tutor Bot, and your role is...",1,3
3,"You are a chatbot named FitBot, and your prima...",1,7
4,"You are now Dr. Digital, a healthcare consulti...",1,3


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 12809
Test size: 3203


In [7]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [8]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [10]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 3203/3203 [00:00<00:00, 6611.39 examples/s]


In [11]:
tokenized_train = tokenized_train.remove_columns(["text", "Degree", "__index_level_0__"])
tokenized_test = tokenized_test.remove_columns(["text", "Degree", "__index_level_0__"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

In [13]:
training_args = TrainingArguments(
    output_dir="../results/distilbert_model",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="../results/distilbert_logs",
    logging_steps=100
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()